# 🤖 ROTBOTS — Full Pipeline
## Topic → Video in One Notebook

---

All 9 pipeline stages in one place. Run cells top to bottom.

> **🤔** Every cell is an automated decision. What does the machine choose?

---

In [ ]:
# SETUP — run once
!pip install -q requests beautifulsoup4 pymupdf edge-tts fal-client yt-dlp faster-whisper
import json, re, random, requests, subprocess, shutil, os, time as _time, threading, edge_tts
from pathlib import Path
from bs4 import BeautifulSoup
from IPython.display import display, Markdown, HTML, Audio, Video
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/MyDrive/rotbots_workshop')
BASE_DIR.mkdir(parents=True, exist_ok=True)
TEMP = Path('/content/temp_assembly'); TEMP.mkdir(exist_ok=True)
def progress_bar(c,t,l='',w=30):
    p=c/t if t>0 else 0;f=int(w*p);return f'<div style="font-family:monospace;font-size:14px;padding:2px 0;">{chr(9608)*f}{chr(9617)*(w-f)} {c}/{t} {l} ({p:.0%})</div>'
def progress_html(title,c,t,l='',d='',color='#4ecca3'):
    return f'<div style="background:#1a1a2e;padding:12px;border-radius:8px;color:#eaeaea;font-family:monospace;"><div style="font-size:16px;font-weight:bold;color:{color};">{title}</div>{progress_bar(c,t,l)}' + (f'<div style="color:#a0a0a0;font-size:12px;margin-top:4px;">{d}</div>' if d else '') + '</div>'
print('\u2705 Ready')

In [ ]:
# API KEYS
GROQ_API_KEY = ''  # https://console.groq.com/keys
FAL_API_KEY = ''   # https://fal.ai/dashboard/keys
LLM_MODEL = 'llama-3.3-70b-versatile'
LLM_TEMPERATURE = 0.8
LLM_MAX_TOKENS = 4096
GROQ_API_URL = 'https://api.groq.com/openai/v1/chat/completions'
if GROQ_API_KEY: print('\u2705 Groq')
else: print('\u26a0\ufe0f Paste GROQ_API_KEY')
if FAL_API_KEY: os.environ['FAL_KEY'] = FAL_API_KEY; print('\u2705 fal.ai')
else: print('\u26a0\ufe0f Paste FAL_API_KEY')

---
# \u2501\u2501 01: Video Plan

In [ ]:
TOPIC = 'The history and ethics of AI-generated art'
SOURCES = ['https://en.wikipedia.org/wiki/Artificial_intelligence_art']
TOTAL_VIDEO_LENGTH = 60
SECONDS_PER_SCENE = 5
ARCHIVE_RATIO = 0.0
UPLOAD_RATIO = 0.0
ENABLE_CREDITS = True
ENABLE_SUBTITLES = False
ENABLE_MUSIC = False
ENABLE_EFFECTS = True
SESSION_NAME = ''

In [ ]:
# CALCULATE PLAN
AI_RATIO = max(0, 1.0 - ARCHIVE_RATIO - UPLOAD_RATIO)
CREDITS_LENGTH = 8 if ENABLE_CREDITS else 0
CONTENT_LENGTH = TOTAL_VIDEO_LENGTH - CREDITS_LENGTH
NARRATION_LENGTH = CONTENT_LENGTH
NUM_TOTAL_SCENES = max(2, int(CONTENT_LENGTH / SECONDS_PER_SCENE))
WORDS_PER_SCENE = SECONDS_PER_SCENE * 2.5

# Scene counts: 0-ratio = 0 scenes, AI gets remainder
NUM_ARCHIVE_SCENES = int(NUM_TOTAL_SCENES * ARCHIVE_RATIO) if ARCHIVE_RATIO > 0 else 0
NUM_UPLOAD_SCENES = int(NUM_TOTAL_SCENES * UPLOAD_RATIO) if UPLOAD_RATIO > 0 else 0
NUM_AI_SCENES = max(1, NUM_TOTAL_SCENES - NUM_ARCHIVE_SCENES - NUM_UPLOAD_SCENES)
while NUM_AI_SCENES + NUM_ARCHIVE_SCENES + NUM_UPLOAD_SCENES > NUM_TOTAL_SCENES:
    if NUM_ARCHIVE_SCENES > 0: NUM_ARCHIVE_SCENES -= 1
    elif NUM_UPLOAD_SCENES > 0: NUM_UPLOAD_SCENES -= 1
    else: break

NUM_CHAPTERS = max(1, min(3, NUM_TOTAL_SCENES // 3))
SECTIONS_PER_CHAPTER = max(1, NUM_TOTAL_SCENES // NUM_CHAPTERS)

# Interleave scene order
scene_types = []
ai_p = 0; ar_p = 0; up_p = 0
for i in range(NUM_TOTAL_SCENES):
    remaining = max(1, NUM_TOTAL_SCENES - i)
    ai_need = (NUM_AI_SCENES - ai_p) / remaining
    ar_need = (NUM_ARCHIVE_SCENES - ar_p) / remaining
    up_need = (NUM_UPLOAD_SCENES - up_p) / remaining
    if ar_need >= ai_need and ar_need >= up_need and ar_p < NUM_ARCHIVE_SCENES:
        scene_types.append('archive'); ar_p += 1
    elif up_need >= ai_need and up_p < NUM_UPLOAD_SCENES:
        scene_types.append('upload'); up_p += 1
    else:
        scene_types.append('ai_generated'); ai_p += 1

if not SESSION_NAME:
    SESSION_NAME = '-'.join(re.sub(r'[^a-zA-Z0-9 ]', '', TOPIC.lower()).split()[:4])
SESSION_DIR = BASE_DIR / SESSION_NAME
for d in ['', 'videos', 'audio', 'archive_clips', 'uploads']:
    (SESSION_DIR / d).mkdir(parents=True, exist_ok=True)
VIDEOS_DIR = SESSION_DIR / 'videos'; AUDIO_DIR = SESSION_DIR / 'audio'

plan = dict(topic=TOPIC, sources=SOURCES, session_name=SESSION_NAME,
    total_video_length=TOTAL_VIDEO_LENGTH, seconds_per_scene=SECONDS_PER_SCENE,
    content_length=CONTENT_LENGTH, credits_length=CREDITS_LENGTH,
    narration_length=NARRATION_LENGTH, ai_ratio=AI_RATIO,
    archive_ratio=ARCHIVE_RATIO, upload_ratio=UPLOAD_RATIO,
    num_total_scenes=NUM_TOTAL_SCENES, num_ai_scenes=NUM_AI_SCENES,
    num_archive_scenes=NUM_ARCHIVE_SCENES, num_upload_scenes=NUM_UPLOAD_SCENES,
    scene_types=scene_types, enable_credits=ENABLE_CREDITS,
    enable_subtitles=ENABLE_SUBTITLES, enable_music=ENABLE_MUSIC,
    enable_effects=ENABLE_EFFECTS)
with open(SESSION_DIR / 'video_plan.json', 'w') as f: json.dump(plan, f, indent=2)
with open(SESSION_DIR / 'session_info.json', 'w') as f: json.dump(dict(name=SESSION_NAME, topic=TOPIC), f, indent=2)

from collections import Counter
sc_counts = Counter(scene_types)
print(f'Session: {SESSION_NAME}')
print(f'Plan: {TOTAL_VIDEO_LENGTH}s = {CONTENT_LENGTH}s content + {CREDITS_LENGTH}s credits')
print(f'Scenes: {NUM_TOTAL_SCENES} total = {sc_counts.get("ai_generated",0)} AI + {sc_counts.get("archive",0)} archive + {sc_counts.get("upload",0)} upload')
icons = {"ai_generated": "\ud83e\udd16", "archive": "\ud83c\udfdb\ufe0f", "upload": "\ud83d\udcc2"}
print(' '.join(icons.get(t,'?') for t in scene_types))

---
# \u2501\u2501 02: Script Writer

In [ ]:
# HELPERS
def query_llm(prompt, system_prompt=None, temperature=None):
    messages = []
    if system_prompt: messages.append(dict(role='system', content=system_prompt))
    messages.append(dict(role='user', content=prompt))
    r = requests.post(GROQ_API_URL, headers={'Authorization': f'Bearer {GROQ_API_KEY}', 'Content-Type': 'application/json'}, json=dict(model=LLM_MODEL, messages=messages, temperature=temperature or LLM_TEMPERATURE, max_tokens=LLM_MAX_TOKENS), timeout=120)
    r.raise_for_status()
    c = r.json()['choices'][0]['message']['content']
    if '<think>' in c and '</think>' in c: c = c.split('</think>')[-1].strip()
    return c

def parse_json_response(r):
    r = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', '', r.strip())
    if r.startswith('```'): r = r.split('\n', 1)[-1].rsplit('```', 1)[0]
    r = r.strip()
    for ch in ['{', '[']:
        idx = r.find(ch)
        if idx > 0: r = r[idx:]; break
    for end in ['}', ']']:
        li = r.rfind(end)
        if li != -1:
            t = r[:li+1]
            for text in [t, re.sub(r',\s*([}\]])', r'\1', t)]:
                try: return json.loads(text, strict=False)
                except: pass
    return json.loads(re.sub(r',\s*([}\]])', r'\1', r), strict=False)

def fetch_website_text(url, max_chars=10000):
    r = requests.get(url.strip(), headers={'User-Agent': 'Mozilla/5.0'}, timeout=30)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, 'html.parser')
    for el in soup(['script','style','nav','header','footer','aside','form']): el.decompose()
    main = None
    for sel in ['article', 'main', '[role="main"]', '.content', '#content']:
        main = soup.select_one(sel)
        if main: break
    text = (main or soup.find('body') or soup).get_text(separator=' ', strip=True)
    return re.sub(r'\s+', ' ', text).strip()[:max_chars]

print('\u2705 Helpers loaded')

In [ ]:
# SCRAPE + SUMMARIZE
prog = display(HTML(''), display_id=True); source_texts = []; t0 = _time.time()
for i, source in enumerate(SOURCES):
    prog.update(HTML(progress_html(f'Scraping {i+1}/{len(SOURCES)}', i, len(SOURCES)*2, 'steps', source[:60])))
    try: source_texts.append(dict(source=source[:100], text=fetch_website_text(source) if source.startswith('http') else source))
    except Exception as e: print(f'   Error: {e}')
summaries = []
for i, src in enumerate(source_texts):
    prog.update(HTML(progress_html(f'Summarizing {i+1}/{len(source_texts)}', len(SOURCES)+i, len(SOURCES)*2, 'steps')))
    summaries.append(dict(source=src['source'], summary=query_llm(f'Summarize in 2-3 paragraphs for video essay about "{TOPIC}":\n\n{src["text"][:6000]}', 'Be concise.')))
with open(SESSION_DIR / 'summaries.json', 'w') as f: json.dump(dict(topic=TOPIC, sources=summaries), f, indent=2)
prog.update(HTML(progress_html(f'Done: {len(summaries)} sources ({_time.time()-t0:.1f}s)', len(SOURCES)*2, len(SOURCES)*2, 'steps')))

In [ ]:
# OUTLINE + CHAPTERS
prog = display(HTML(''), display_id=True); t0 = _time.time(); total_llm = 1 + NUM_CHAPTERS
prog.update(HTML(progress_html('Generating outline...', 0, total_llm, 'LLM')))
st = '\n\n'.join([f'SOURCE: {s["source"]}\n{s["summary"]}' for s in summaries])
for attempt in range(3):
    try:
        outline = parse_json_response(query_llm(f'Essay outline for {NARRATION_LENGTH}s video: "{TOPIC}"\n\nRESEARCH:\n{st}\n\nExactly {NUM_CHAPTERS} chapters. JSON only: {{"title":"...","thesis":"...","chapters":[{{"chapter":1,"title":"...","summary":"...","key_points":["..."]}}]}}', 'Expert essay writer.'))
        break
    except: pass
if len(outline.get('chapters', [])) > NUM_CHAPTERS: outline['chapters'] = outline['chapters'][:NUM_CHAPTERS]
for i, ch in enumerate(outline['chapters']):
    prog.update(HTML(progress_html(f'Writing Ch {ch["chapter"]}: {ch["title"]}', i+1, total_llm, 'LLM')))
    try:
        sections = parse_json_response(query_llm(f'Write {SECTIONS_PER_CHAPTER} sections. MAX {int(WORDS_PER_SCENE)} words each.\nTOPIC: {TOPIC}\nCHAPTER: {ch["title"]}\nJSON: [{{"section":1,"narration":"...","visual_direction":"...","mood":"..."}}]', f'Scriptwriter. MAX {int(WORDS_PER_SCENE)} words per section.'))
        if isinstance(sections, dict):
            for v in sections.values():
                if isinstance(v, list): sections = v; break
            else: sections = [sections]
    except: sections = [dict(section=1, narration=ch['title'], visual_direction='', mood='neutral')]
    outline['chapters'][i]['sections'] = sections[:SECTIONS_PER_CHAPTER]
outline['credits'] = dict(sources=[s['source'] for s in summaries])
with open(SESSION_DIR / 'essay_script.json', 'w') as f: json.dump(outline, f, indent=2)
tw = sum(len(s.get('narration','').split()) for c in outline['chapters'] for s in c.get('sections', []))
prog.update(HTML(progress_html(f'Done: "{outline.get("title","")}" {tw}w', total_llm, total_llm, 'LLM')))

In [ ]:
# STORYBOARD + STYLES
STYLE_ARCS = dict(
    calm_to_dramatic=dict(early=['documentary','nature'], middle=['news_report','sports_commentary'], late=['action_movie','horror']),
    documentary_journey=dict(early=['documentary'], middle=['nature','news_report'], late=['action_movie','sports_commentary']),
    chaos_arc=dict(early=['classic_brainrot','zoomer_brainrot'], middle=['surreal_brainrot','hyperpop_brainrot'], late=['fever_dream_brainrot','cursed_brainrot']),
)
STYLES = dict(
    documentary=dict(vk='cinematic, professional lighting'), nature=dict(vk='wide nature shots, golden hour'),
    news_report=dict(vk='news studio, professional'), sports_commentary=dict(vk='dynamic angles, slow motion'),
    action_movie=dict(vk='dynamic movement, dramatic lighting'), horror=dict(vk='dark lighting, shadows, fog'),
    classic_brainrot=dict(vk='chaotic, deep fried'), surreal_brainrot=dict(vk='dreamlike, impossible geometry'),
    zoomer_brainrot=dict(vk='meme aesthetic, ironic'), hyperpop_brainrot=dict(vk='maximalist, Y2K'),
    fever_dream_brainrot=dict(vk='hallucinatory, warped'), cursed_brainrot=dict(vk='unsettling, jpeg artifacts'),
)
CHOSEN_ARC = 'calm_to_dramatic'
arc = STYLE_ARCS[CHOSEN_ARC]

all_sec = []
for ch in outline.get('chapters', []):
    for sec in ch.get('sections', []):
        all_sec.append(dict(**sec, chapter_title=ch['title'], chapter=ch.get('chapter', 0)))

scenes = []; sn = 1; si = 0
for stype in scene_types:
    sec = all_sec[si] if si < len(all_sec) else dict(narration='', visual_direction='', mood='neutral', chapter_title=TOPIC, section=si+1)
    scenes.append(dict(scene=sn, scene_type=stype, title=f'{sec.get("chapter_title","")} - Part {sec.get("section", si+1)}', narration=sec.get('narration', ''), visual_direction=sec.get('visual_direction', ''), mood=sec.get('mood', ''), duration=SECONDS_PER_SCENE))
    sn += 1; si += 1

ai_sc = [s for s in scenes if s['scene_type'] == 'ai_generated']
total_ai = len(ai_sc)
ee = max(1, int(total_ai * 0.25)); ls = max(ee+1, int(total_ai * 0.75)); last = None
for i, sc in enumerate(ai_sc):
    pool = arc.get('early' if i < ee else 'late' if i >= ls else 'middle', ['documentary'])
    av = [s for s in pool if s != last] or pool; st = random.choice(av)
    sc['assigned_style'] = st; sc['visual_keywords'] = STYLES.get(st, {}).get('vk', ''); last = st

with open(SESSION_DIR / 'storyboard.json', 'w') as f: json.dump(scenes, f, indent=2)
print(f'Storyboard: {len(scenes)} scenes')
for s in scenes:
    icon = {"ai_generated": "\ud83e\udd16", "archive": "\ud83c\udfdb\ufe0f", "upload": "\ud83d\udcc2"}.get(s['scene_type'], '?')
    style_tag = f' [{s.get("assigned_style","")}]' if s.get('assigned_style') else ''
    print(f'   {icon} Scene {s["scene"]}: {s["title"]} ({s["scene_type"]}){style_tag}')

In [ ]:
# T2V PROMPTS (AI scenes only)
OPENINGS = ['Start with SHOT TYPE', 'Start with ACTION', 'Start with ENVIRONMENT']
prompts = []; prog = display(HTML(''), display_id=True); t0 = _time.time()
ai_scenes_list = [s for s in scenes if s['scene_type'] == 'ai_generated']
if not ai_scenes_list:
    print('WARNING: No AI scenes in plan! Check ARCHIVE_RATIO/UPLOAD_RATIO.')
for idx, sc in enumerate(ai_scenes_list):
    st = sc.get('assigned_style', 'documentary')
    prog.update(HTML(progress_html(f'Prompt {idx+1}/{len(ai_scenes_list)}: Scene {sc["scene"]}', idx, len(ai_scenes_list), 'prompts', f'Style: {st}')))
    t2v = query_llm(f'T2V prompt for {SECONDS_PER_SCENE}s:\nVisual: {sc.get("visual_direction","")}\nMood: {sc.get("mood","")}\n{random.choice(OPENINGS)}\nOutput ONLY the prompt text.', f'T2V prompt expert. ONE paragraph, 3-5 sentences for a {SECONDS_PER_SCENE}s video clip. Style: {st}. Visual keywords: {sc.get("visual_keywords","")}. No text overlays.')
    prompts.append(dict(scene=sc['scene'], title=sc['title'], t2v_prompt=t2v.strip().strip('"'), style=st, narration=sc.get('narration',''), duration=SECONDS_PER_SCENE))
with open(SESSION_DIR / 'prompts.json', 'w') as f: json.dump(prompts, f, indent=2)
prog.update(HTML(progress_html(f'Done: {len(prompts)} prompts ({_time.time()-t0:.1f}s)', len(ai_scenes_list), max(1, len(ai_scenes_list)), 'prompts')))

---
# \u2501\u2501 03: Archive Scraper (optional)

Skip if `ARCHIVE_RATIO = 0.0`.

In [ ]:
# ARCHIVE URLS
ARCHIVE_URLS = []
PREFER_HEIGHT = 480; MAX_CLIP_DURATION = 10; MIN_CLIP_DURATION = 3; MAX_EXTRACT_DURATION = 180

def parse_archive_id(url):
    url = url.strip().rstrip('/')
    m = re.search(r'archive\.org/(?:details|download)/([^/?#]+)', url)
    if m: return m.group(1)
    if '/' not in url and '.' not in url: return url
    raise ValueError(f'Cannot parse: {url}')

def get_dur(path):
    try: return float(subprocess.run(['ffprobe','-v','quiet','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',str(path)], capture_output=True, text=True, timeout=10).stdout.strip())
    except: return 0

archive_clips = []
ARCHIVE_DIR = SESSION_DIR / 'archive_clips'; ARCHIVE_DIR.mkdir(exist_ok=True)
if ARCHIVE_URLS and NUM_ARCHIVE_SCENES > 0:
    ATEMP = Path('/content/temp_archive'); ATEMP.mkdir(exist_ok=True)
    for i, url in enumerate(ARCHIVE_URLS):
        try: aid = parse_archive_id(url)
        except ValueError as e: print(f'Error: {e}'); continue
        print(f'Downloading [{i+1}/{len(ARCHIVE_URLS)}] {aid}')
        out_tpl = str(ATEMP / f'{aid}.%(ext)s')
        try:
            subprocess.run(['yt-dlp', f'https://archive.org/details/{aid}', '-f', f'bestvideo[height<={PREFER_HEIGHT}]+bestaudio/best[height<={PREFER_HEIGHT}]/best', '--merge-output-format', 'mp4', '--no-playlist', '--no-warnings', '-o', out_tpl, '--quiet'], check=True, timeout=600)
        except: print('   Download failed'); continue
        video = None
        for fp in ATEMP.iterdir():
            if fp.stem == aid and fp.suffix in ('.mp4','.mkv','.webm'): video = fp; break
        if not video: continue
        total = get_dur(video); clip_dir = ARCHIVE_DIR / aid; clip_dir.mkdir(exist_ok=True)
        extract_end = min(total, MAX_EXTRACT_DURATION) if MAX_EXTRACT_DURATION > 0 else total
        t = 0; idx = 0
        while t < extract_end:
            clip_dur = min(MAX_CLIP_DURATION, extract_end - t)
            if clip_dur < MIN_CLIP_DURATION: break
            clip_out = clip_dir / f'archive_{idx:03d}.mp4'
            r = subprocess.run(['ffmpeg','-y','-ss',str(t),'-i',str(video),'-t',str(clip_dur),'-c:v','libx264','-preset','fast','-crf','23','-an',str(clip_out)], capture_output=True, text=True, timeout=120)
            if r.returncode == 0 and clip_out.exists():
                archive_clips.append(dict(path=str(clip_out), duration=round(clip_dur,1), archive_id=aid)); idx += 1
            t += clip_dur
        video.unlink(missing_ok=True); print(f'   {idx} clips')
    with open(SESSION_DIR / 'archive_clips.json', 'w') as f: json.dump(dict(clips=archive_clips, total=len(archive_clips)), f, indent=2)
    print(f'{len(archive_clips)} total archive clips')
elif NUM_ARCHIVE_SCENES > 0: print(f'WARNING: Plan needs {NUM_ARCHIVE_SCENES} archive clips but no ARCHIVE_URLS!')
else: print('No archive footage needed')

---
# \u2501\u2501 04: Upload Footage (optional)

Skip if `UPLOAD_RATIO = 0.0`.

In [ ]:
upload_clips = []
if NUM_UPLOAD_SCENES > 0:
    from google.colab import files
    UPLOADS_DIR = SESSION_DIR / 'uploads'; UPLOADS_DIR.mkdir(exist_ok=True)
    print(f'Upload {NUM_UPLOAD_SCENES} MP4 clips:')
    uploaded = files.upload()
    for fn, content in uploaded.items():
        dest = UPLOADS_DIR / fn
        with open(dest, 'wb') as f: f.write(content)
        upload_clips.append(dict(path=str(dest), filename=fn))
        print(f'   {fn} ({len(content)//1024}KB)')
    with open(SESSION_DIR / 'upload_clips.json', 'w') as f: json.dump(dict(clips=upload_clips, total=len(upload_clips)), f, indent=2)
else: print('No uploads needed (UPLOAD_RATIO = 0)')

---
# \u2501\u2501 05: Effects & Chain Log

In [ ]:
EFFECT_MODE = 'random'; EFFECT_INTENSITY = 0.7
if ENABLE_EFFECTS and prompts:
    enames = ['film_grain','vhs_artifacts','celluloid_scratches','sepia_tone','bw_transition','color_grade_warm','color_grade_cool','vignette','flicker','desaturate']
    for p in prompts: p['ffmpeg_effects'] = [random.choice(enames)]; p['effect_intensity'] = EFFECT_INTENSITY
    with open(SESSION_DIR / 'prompts.json', 'w') as f: json.dump(prompts, f, indent=2)
    for p in prompts: print(f'   Scene {p["scene"]}: {p["ffmpeg_effects"][0]}')
elif not prompts: print('No AI prompts to apply effects to')
else: print('Effects disabled')

chain = dict(session=SESSION_NAME, plan=plan, interactions=[])
for s in summaries: chain['interactions'].append(dict(stage='summarize', source=s['source'][:40]))
chain['interactions'].append(dict(stage='outline', title=outline.get('title','')))
for ch in outline.get('chapters',[]):
    for sec in ch.get('sections',[]): chain['interactions'].append(dict(stage='chapter', ref=f'Ch{ch["chapter"]}.{sec.get("section","?")}'))
for p in prompts: chain['interactions'].append(dict(stage='t2v_prompt', scene=p['scene'], effect=p.get('ffmpeg_effects',[])))
with open(SESSION_DIR / 'chain_log.json', 'w') as f: json.dump(chain, f, indent=2)
print(f'{len(chain["interactions"])} AI decisions logged')

---
# \u2501\u2501 06: The Voice

In [ ]:
VOICES = dict(documentary='en-US-GuyNeural', female_us='en-US-AriaNeural', male_uk='en-GB-RyanNeural', dramatic='en-GB-RyanNeural')
CHOSEN_VOICE = 'documentary'; voice_name = VOICES[CHOSEN_VOICE]

async def gen_tts(text, path, voice, rate='+0%'):
    await edge_tts.Communicate(text, voice, rate=rate).save(str(path))

prog = display(HTML(''), display_id=True); audio_files = []; t0 = _time.time()
narrations = [dict(scene=s['scene'], narration=s['narration']) for s in scenes if s.get('narration')]
for idx, n in enumerate(narrations):
    out = AUDIO_DIR / f'narration_{n["scene"]:03d}.mp3'
    prog.update(HTML(progress_html(f'TTS Scene {n["scene"]}', idx, len(narrations), 'narrations')))
    try:
        await gen_tts(n['narration'], out, voice_name)
        dur_val = get_dur(out) or 5.0
        audio_files.append(dict(scene=n['scene'], path=str(out), duration=dur_val, text=n['narration']))
    except Exception as e: print(f'   Error: {e}')

dur_map = {a['scene']: a['duration'] for a in audio_files}
for p in prompts:
    if p['scene'] in dur_map: p['duration'] = max(3, min(10, int(dur_map[p['scene']]) + 1))
with open(SESSION_DIR / 'prompts.json', 'w') as f: json.dump(prompts, f, indent=2)
with open(SESSION_DIR / 'audio_manifest.json', 'w') as f: json.dump(dict(voice=voice_name, files=audio_files), f, indent=2)
td = sum(a['duration'] for a in audio_files)
prog.update(HTML(progress_html(f'Done: {len(audio_files)} narrations, {td:.1f}s', len(narrations), len(narrations), 'narrations')))

---
# \u2501\u2501 07: AI Video Generation

In [ ]:
import fal_client
MODELS = dict(**{'wan-t2v': dict(endpoint='fal-ai/wan-t2v', ds=5), 'ltx-video': dict(endpoint='fal-ai/ltx-video/v2.1', ds=5)})
CHOSEN_MODEL = 'wan-t2v'; model = MODELS[CHOSEN_MODEL]

if not prompts:
    print('No AI prompts to generate. Check ARCHIVE_RATIO/UPLOAD_RATIO.')
else:
    prog = display(HTML(''), display_id=True); generated_videos = []; t0 = _time.time(); _stop = False
    def _timer(ph, si, done, total, ts):
        sp = list('|/-\\'); tick = 0
        while not _stop:
            el = _time.time()-ts; ce = _time.time()-si.get('cs',ts)
            avg = el/done[0] if done[0]>0 else 0; eta = (total-done[0]-1)*avg+max(0,avg-ce) if avg>0 else 0
            s = sp[tick%len(sp)]; p = done[0]/total if total>0 else 0; fl = int(30*p)
            try: ph.update(HTML(progress_html(f'{s} Scene {si.get("sn","?")}: {si.get("title","")}', done[0], total, 'videos', f'clip: {ce:.0f}s, ETA: {eta:.0f}s', '#e94560')))
            except: pass
            tick += 1; _time.sleep(1)
    for i, pd in enumerate(prompts):
        sn = pd['scene']; si = dict(sn=sn, title=pd['title'], cs=_time.time()); done = [len(generated_videos)]
        _stop = False; t = threading.Thread(target=_timer, args=(prog, si, done, len(prompts), t0), daemon=True); t.start()
        try:
            inp = dict(prompt=pd['t2v_prompt'], aspect_ratio='16:9', enable_prompt_expansion=False)
            result = fal_client.subscribe(model['endpoint'], arguments=inp); url = None
            if isinstance(result, dict):
                if 'video' in result and isinstance(result['video'], dict): url = result['video'].get('url')
                elif 'video' in result: url = result['video']
                elif 'url' in result: url = result['url']
            if url:
                vp = VIDEOS_DIR / f'scene_{sn:03d}.mp4'
                with open(vp, 'wb') as f: f.write(requests.get(url, timeout=120).content)
                generated_videos.append(dict(scene=sn, path=str(vp), duration=pd.get('duration',5), source='generated'))
        except Exception as e: print(f'   Error Scene {sn}: {e}')
        finally: _stop = True; t.join(timeout=2)
    el = _time.time()-t0
    prog.update(HTML(progress_html(f'Done: {len(generated_videos)}/{len(prompts)} videos in {el:.0f}s', len(prompts), len(prompts), 'videos')))
    with open(SESSION_DIR / 'video_manifest.json', 'w') as f: json.dump(dict(model=CHOSEN_MODEL, videos=generated_videos), f, indent=2)

---
# \u2501\u2501 08: Subtitles (optional)

Skip if `ENABLE_SUBTITLES = False`.

In [ ]:
sub_file = SESSION_DIR / 'subtitles.ass'
if ENABLE_SUBTITLES:
    from faster_whisper import WhisperModel
    wm = WhisperModel('base', device='cpu', compute_type='int8')
    afs = sorted(AUDIO_DIR.glob('narration_*.mp3'))
    all_words = []; toff = 0.0
    for af in afs:
        segs, info = wm.transcribe(str(af), word_timestamps=True, language='en')
        for seg in segs:
            if seg.words:
                for w in seg.words: all_words.append(dict(word=w.word.strip(), start=w.start+toff, end=w.end+toff))
        try: fdur = float(subprocess.run(['ffprobe','-v','quiet','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',str(af)], capture_output=True, text=True, timeout=10).stdout.strip())
        except: fdur = 5.0
        toff += fdur
    W=1280; H=720; sc_f=H/512
    sty = dict(font='Impact', size=int(80*sc_f), bold=-1, outline=int(7*sc_f), shadow=0, mv=int(20*sc_f))
    hdr = f'[Script Info]\nTitle: ROTBOTS\nScriptType: v4.00+\nWrapStyle: 0\nScaledBorderAndShadow: yes\nPlayResX: {W}\nPlayResY: {H}\n\n[V4+ Styles]\nFormat: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding\nStyle: TT,{sty["font"]},{sty["size"]},&H00FFFFFF,&H0000FFFF,&H00000000,&H00000000,{sty["bold"]},0,0,0,100,100,2,0,1,{sty["outline"]},{sty["shadow"]},5,10,10,{sty["mv"]},1\n\n[Events]\nFormat: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text\n'
    def fmt(sec): h=int(sec//3600);m=int((sec%3600)//60);s=int(sec%60);cs=int((sec%1)*100); return f'{h}:{m:02d}:{s:02d}.{cs:02d}'
    anim = r'{\fscx0\fscy0\t(0,40,\fscx130\fscy130)\t(40,80,\fscx100\fscy100)}'
    dlg = []
    for w in all_words:
        wd = w['word'].upper()
        dlg.append(f'Dialogue: 0,{fmt(w["start"])},{fmt(w["end"])},TT,,0,0,0,,{anim}{wd}')
    with open(sub_file, 'w', encoding='utf-8') as f: f.write(hdr + '\n'.join(dlg))
    print(f'subtitles.ass: {len(dlg)} words')
else: print('Subtitles disabled')

---
# \u2501\u2501 09: Assemble

In [ ]:
def dur(p):
    try: return float(subprocess.run(['ffprobe','-v','quiet','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',str(p)], capture_output=True, text=True, timeout=10).stdout.strip())
    except: return 5.0

def ff(cmd, desc=''):
    if desc: print(f'   {desc}...', end=' ', flush=True)
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=600)
    if r.returncode == 0:
        if desc: print('OK')
        return True
    if desc: print('FAIL')
    if r.stderr: print(f'   {r.stderr[-200:]}')
    return False

def build_effect_filter(name, intensity=0.7):
    i = max(0.0, min(1.0, intensity))
    effects = dict(film_grain=f'noise=alls={int(12*i)}:allf=t', vhs_artifacts=f'noise=alls={int(30*i)}:allf=t,rgbashift=rh={int(2*i)}:bh={int(-2*i)}', celluloid_scratches=f'noise=c0s={int(15*i)}:c0f=t', color_grade_warm=f'eq=saturation={1+0.1*i:.3f}:brightness={0.02*i:.3f}', color_grade_cool=f'eq=saturation={1-0.1*i:.3f},colorbalance=bs={0.12*i:.3f}', vignette=f'vignette=PI/4*{i:.3f}', flicker=f'noise=alls={int(8*i)}:allf=t', desaturate=f'eq=saturation={0.5+0.5*(1-i):.3f}')
    if name == 'sepia_tone': inv=1-i; return f'colorchannelmixer={inv+i*0.393:.3f}:{i*0.769:.3f}:{i*0.189:.3f}:0:{i*0.349:.3f}:{inv+i*0.686:.3f}:{i*0.168:.3f}:0:{i*0.272:.3f}:{i*0.534:.3f}:{inv+i*0.131:.3f}:0'
    if name == 'bw_transition': inv=1-i; return f'colorchannelmixer={inv+i*0.3:.3f}:{i*0.4:.3f}:{i*0.3:.3f}:0:{i*0.3:.3f}:{inv+i*0.4:.3f}:{i*0.3:.3f}:0:{i*0.3:.3f}:{i*0.4:.3f}:{inv+i*0.3:.3f}:0'
    return effects.get(name)

In [ ]:
# BUILD SCENE SEQUENCE
effects_map = {int(p['scene']): p for p in prompts if p.get('ffmpeg_effects')}
print(f'Building {len(scenes)} scenes...')
norm = []; arc_idx = 0; upl_idx = 0

for sc in scenes:
    sn = sc['scene']; stype = sc['scene_type']
    icon = {"ai_generated": "AI", "archive": "ARC", "upload": "UPL"}.get(stype, '?')
    out = TEMP / f'scene_{sn:03d}.mp4'; src = None
    if stype == 'ai_generated':
        c = VIDEOS_DIR / f'scene_{sn:03d}.mp4'
        if c.exists(): src = c
        else: print(f'   [{icon}] Scene {sn}: MISSING (run 07)'); continue
    elif stype == 'archive':
        if arc_idx < len(archive_clips): src = Path(archive_clips[arc_idx]['path']); arc_idx += 1
        if not src or not src.exists(): print(f'   [{icon}] Scene {sn}: no archive clip'); continue
    elif stype == 'upload':
        if upl_idx < len(upload_clips): src = Path(upload_clips[upl_idx]['path']); upl_idx += 1
        if not src or not src.exists(): print(f'   [{icon}] Scene {sn}: no upload clip'); continue
    vf = 'scale=1280:720:force_original_aspect_ratio=decrease,pad=1280:720:(ow-iw)/2:(oh-ih)/2:black'
    if stype == 'ai_generated' and sn in effects_map:
        for eff in effects_map[sn].get('ffmpeg_effects', []):
            ef = build_effect_filter(eff, effects_map[sn].get('effect_intensity', 0.7))
            if ef: vf += ',' + ef
    if ff(['ffmpeg','-y','-i',str(src),'-vf',vf,'-r','24','-pix_fmt','yuv420p','-c:v','libx264','-preset','fast','-crf','23','-an','-t',str(sc.get('duration',5)),str(out)], f'[{icon}] Scene {sn}'):
        norm.append(out)
print(f'{len(norm)} scenes ready')

In [ ]:
# CREDITS + CONCAT + AUDIO + SUBS -> FINAL
credits_path = None
if ENABLE_CREDITS:
    essay = json.load(open(SESSION_DIR / 'essay_script.json')) if (SESSION_DIR / 'essay_script.json').exists() else None
    title = essay.get('title', 'Untitled') if essay else 'Untitled'
    src_list = essay.get('credits',{}).get('sources',[]) if essay else []
    lines = [title, '', 'Generated by ROTBOTS', ''] + [s[:60] for s in src_list] + ['', 'LI-MA TDA 2026']
    D = 8; LH = 42; spd = (720 + len(lines)*LH) / D
    flt = []
    for i, l in enumerate(lines):
        if not l: continue
        safe_l = l.replace("'", "\u2019").replace(":", "\\:")
        flt.append(f"drawtext=text='{safe_l}':fontsize=28:fontcolor=white:x=(w-text_w)/2:y=h+{i*LH}-{spd:.1f}*t")
    credits_path = TEMP / 'credits.mp4'
    ff(['ffmpeg','-y','-f','lavfi','-i',f'color=c=black:s=1280x720:d={D}:r=24','-vf',','.join(flt),'-pix_fmt','yuv420p','-c:v','libx264','-preset','fast',str(credits_path)], 'Credits')

cf = TEMP / 'concat.txt'
with open(cf, 'w') as f:
    for v in norm: f.write(f"file '{v}'\n")
    if credits_path and credits_path.exists(): f.write(f"file '{credits_path}'\n")
concat_out = TEMP / 'concatenated.mp4'
ff(['ffmpeg','-y','-f','concat','-safe','0','-i',str(cf),'-c','copy',str(concat_out)], 'Concat')
video_duration = dur(concat_out)
print(f'   Video: {video_duration:.1f}s')

audio_out = TEMP / 'with_audio.mp4'
audio_files_sorted = sorted(AUDIO_DIR.glob('narration_*.mp3')) if AUDIO_DIR.exists() else []
if audio_files_sorted:
    acf = TEMP / 'ac.txt'
    with open(acf, 'w') as f:
        for a in audio_files_sorted: f.write(f"file '{a}'\n")
    narr = TEMP / 'narration.mp3'
    ff(['ffmpeg','-y','-f','concat','-safe','0','-i',str(acf),'-c','copy',str(narr)], 'Concat narration')
    narr_padded = TEMP / 'narration_padded.mp3'
    ff(['ffmpeg','-y','-i',str(narr),'-af',f'apad=whole_dur={video_duration}','-c:a','libmp3lame','-b:a','128k',str(narr_padded)], 'Pad audio')
    ff(['ffmpeg','-y','-i',str(concat_out),'-i',str(narr_padded),'-c:v','copy','-c:a','aac','-b:a','192k','-map','0:v:0','-map','1:a:0',str(audio_out)], 'Merge audio')
else: shutil.copy2(str(concat_out), str(audio_out))

final = SESSION_DIR / 'final_video.mp4'
if ENABLE_SUBTITLES and sub_file.exists():
    la = TEMP / 'subtitles.ass'; shutil.copy2(str(sub_file), str(la))
    if not ff(['ffmpeg','-y','-i',str(audio_out),'-vf',f'ass={str(la)}','-c:v','libx264','-preset','fast','-crf','23','-c:a','copy',str(final)], 'Burn subs'):
        shutil.copy2(str(audio_out), str(final))
else: shutil.copy2(str(audio_out), str(final))

if final.exists(): print(f'\nFinal: {dur(final):.1f}s, {final.stat().st_size/(1024*1024):.1f}MB')

---
## Watch

In [ ]:
essay = json.load(open(SESSION_DIR / 'essay_script.json')) if (SESSION_DIR / 'essay_script.json').exists() else None
title = essay.get('title', 'Untitled') if essay else 'Untitled'
if final.exists():
    display(Markdown(f'# {title}\n*Generated by ROTBOTS*'))
    try: display(Video(str(final), embed=True, width=720))
    except: print(f'Drive: rotbots_workshop/{SESSION_NAME}/final_video.mp4')

print('\nPipeline Summary')
ai_count = sum(1 for s in scenes if s['scene_type']=='ai_generated')
arc_count = sum(1 for s in scenes if s['scene_type']=='archive')
upl_count = sum(1 for s in scenes if s['scene_type']=='upload')
print(f'  AI: {ai_count} scenes ({len(prompts)} generated)')
if arc_count: print(f'  Archive: {arc_count} scenes')
if upl_count: print(f'  Upload: {upl_count} scenes')
print(f'  Narrations: {len(audio_files)}')
if final.exists(): print(f'  Final: {dur(final):.1f}s')

---
# Done

**Every step: automated decisions.** What does that mean?

---
*ROTBOTS \u2014 LI-MA TDA 2026, Amsterdam*